In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.sparse import hstack
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, f1_score

RANDOM_STATE = 0

In [ ]:
df = pd.read_csv('/kaggle/input/nlp-getting-started/train.csv')
df_test = pd.read_csv('/kaggle/input/nlp-getting-started/test.csv')

df

In [ ]:
print(df.sample()["text"].values[0])

In [ ]:
print("No Keywords:", df["keyword"].isna().sum())
print("No location:", df["location"].isna().sum())

In [ ]:
keyword, text = df.loc[df["keyword"].notna(), ["keyword", "text"]].sample().values[0]

print("Keyword:", keyword)
print("Text:", text)

In [ ]:
df["keyword"].str.contains("%20").value_counts()

In [ ]:
df["keyword"] = df["keyword"].str.replace('%20', ' ')

In [ ]:
def clean_text(text):
    # remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    # remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    # remove @mentions
    text = re.sub(r'@\w+', '', text)
    ## remove #topics
    #text = re.sub(r'#\w+', '', text)
    # remove non-alphabetical characters but keep white spaces
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    # lower case text
    text = text.lower()
    return ' '.join(text.split())

In [ ]:
df['text_clean'] = df['text'].apply(clean_text)
df_test['text_clean'] = df_test['text'].apply(clean_text)

In [ ]:
keyword, text, text_clean = df.sample()[["keyword", "text", "text_clean"]].values[0]
print("Keyword:\n", keyword)
print("Raw text:\n", text)
print("Clean text:\n", text_clean)

In [ ]:
check_keyword = lambda row: row["keyword"] in row["text_clean"] \
                if pd.notna(row["keyword"]) else False

df["includes_keyword"] = df.apply(check_keyword, axis=1)
df_test["includes_keyword"] = df_test.apply(check_keyword, axis=1)

df["includes_keyword"].value_counts()

In [ ]:
keyword, text, text_clean = df[df["keyword"].notna() & ~df["includes_keyword"]].sample()[["keyword", "text", "text_clean"]].values[0]
print("Keyword:\n", keyword)
print("Raw text:\n", text)
print("Clean text:\n", text_clean)

In [ ]:
add_keyword = lambda row: row["keyword"] + " " + row["text_clean"] \
                            if not row["includes_keyword"] and pd.notna(row["keyword"]) \
                            else row["text_clean"]

df["text_clean_keyword"] = df.apply(add_keyword, axis=1)
df_test["text_clean_keyword"] = df_test.apply(add_keyword, axis=1)

In [ ]:
keyword, text, text_clean, text_clean_keyword = df[df["keyword"].notna() & ~df["includes_keyword"]].sample()[["keyword", "text", "text_clean", "text_clean_keyword"]].values[0]
print("Keyword:\n", keyword)
print("Raw text:\n", text)
print("Clean text:\n", text_clean)
print("Clean text with keyword:\n", text_clean_keyword)

In [ ]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words='english', max_features=15000)
X_tfidf = vectorizer.fit_transform(df["text_clean_keyword"])
X_test_tfidf = vectorizer.transform(df_test["text_clean_keyword"])

df_tfidf = pd.DataFrame(X_tfidf.toarray(), columns=vectorizer.get_feature_names_out())
print(df.loc[0, "text_clean"])
print(df_tfidf.loc[0, df_tfidf.loc[0] != 0])
df_tfidf

In [ ]:
scaler = MinMaxScaler()

df["length"] = df["text"].apply(len)
df_test["length"] = df_test["text"].apply(len)

df["length_scaled"] = scaler.fit_transform(df[["length"]])
df_test["length_scaled"] = scaler.transform(df_test[["length"]])

df[["text", "length", "length_scaled"]]

In [ ]:
X = hstack([X_tfidf, df[["length_scaled"]].values])
X_test = hstack([X_test_tfidf, df_test[["length_scaled"]].values])

y = df["target"].values

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, 
                                                  stratify=y, 
                                                  test_size=0.2, 
                                                  random_state=RANDOM_STATE)

In [ ]:
model = LogisticRegression(random_state=RANDOM_STATE)
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_val)

print(classification_report(y_val, y_pred, target_names=['Not Disaster', 'Disaster']))

In [ ]:
cm = confusion_matrix(y_val, y_pred)
f1 = f1_score(y_val, y_pred, average='weighted')

plt.figure(figsize=(7,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Disaster', 'Disaster'],
            yticklabels=['Not Disaster', 'Disaster'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix\nF1 Score (weighted): {f1:.3f}', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
y_pred = model.predict(X_test)

df_test["prediction"] = y_pred

df_test[["text", "text_clean_keyword", "prediction"]]

In [ ]:
df_submission = pd.read_csv('/kaggle/input/nlp-getting-started/sample_submission.csv')
df_submission["target"] = y_pred

df_submission

In [ ]:
df_submission.to_csv("submission.csv", index=False)